In [ ]:
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

import os
import shutil
from pathlib import Path
from pyspark.sql import functions as F

In [ ]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=64,
    executor_memory="32g",
    executor_cores=2,
    update_configs={
        "spark.sql.shuffle.partitions": "512",  # 6.2B ÷ 500 = 12.4M rows/partition
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.adaptive.coalescePartitions.enabled": "true",
    },
    aws_profile="default"
)

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

In [ ]:
cache_dir = '/data/playground/slamont/teehr/warehouse/loading/backfill_nwm/local-evaluation/local/cache/fetching/nwm/nwm30_short_range/newly_fixed_streamflow_hourly_inst'

In [ ]:
files = list(Path(cache_dir).glob("**/*.parquet"))
total_size = sum(f.stat().st_size for f in files)
print(f"Files: {len(files)}")
print(f"Total size: {total_size / (1024**3):.2f} GB")

In [ ]:
nwm_sdf = ev.spark.read.parquet(cache_dir)
nwm_sdf.count()

In [ ]:
nwm_sdf.selectExpr("min(value_time)").show()

In [ ]:
ev.secondary_timeseries.load_dataframe(nwm_sdf)

In [ ]:
sdf = ev.secondary_timeseries.filter([
    "configuration_name = 'nwm30_short_range'"
]).to_sdf()

In [ ]:
sdf.count()

In [ ]:
sdf.selectExpr("min(value_time)").show()

In [ ]:
spark.stop()